# 応用編15
V3.8新機能:代理ウォレット機能による権限の制限

このノートブックでは、スマートコントラクト・バージョン3.8から利用できる代理ウォレット機能による権限の制限について説明します。

## 代理ウォレット機能の概要

代理ウォレット機能は、あるウォレット（元ウォレット）の権限を別のウォレット（代理ウォレット）に一時的・限定的に委任できる新機能です。主な利用シーンは、スマホ上のウォレットでPC上のアプリにログインするケース（クロスデバイス利用）や、元ウォレットの一部権限のみを代理ウォレットに委任するケースが想定されます。委任にあたって有効期限やアクセス権限の範囲を設定でき、必要に応じて一度限り有効といったモード指定も可能です。

## 準備

設定やライブラリを読み込みます。  

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

adminWalletを元ウォレットとします。 
元ウォレットのユーザIDを確認しておきます。

In [2]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'search', key: 'me' });
console.log(resp.value[0].id);

u86311649


動作確認用のスマートコントラクトをデプロイします。

In [3]:
var cid = await deploySmartContract({ a: 'number'}, function adv15(){
   return getCallerId();
});

## 代理ウォレットの作成

代理ウォレットを作成します。代理ウォレットはPCのブラウザアプリ内に格納される想定です。  
作成方法は、通常のウォレットと同じ方法で作成できます。

In [4]:
var proxyWallet = await api.importSigningWallet('es', await api.generateWalletKey('es'));

proxyWalletに対応するユーザIDがanonymousであることを確認しておきます。

In [5]:
var resp = await rpc.call(proxyWallet, 'c1query', { type: 'search', key: 'me' });
console.log(resp.value[0].id);

anonymous


## 代理ウォレットへの委任（特定コントラクトのみのアクセスに制限）

元ウォレットから代理ウォレットへ、限定した権限のみを委任します。  
access引数に委任する権限を指定します。  
read_tx, write_tx, queryの各項目に、上で作成したコントラクト(cid)を設定します。

In [6]:
var resp = await rpc.call(adminWallet, 'c1walletproxy', {
    addr: proxyWallet.address,
    expiry: Date.now() + 3600*1000, // 委任の有効期限（現在時刻＋1時間）
    access: { read_tx: [cid], write_tx: [cid], query: [cid] },
});
console.log(resp);

{
  txno: 40296,
  txid: 'xAzudTtfDM4nG4hicuvXStcubuJNdYxNM5Hffh5d9THS3',
  status: 'ok',
  value: null
}


代理ウォレットで特定コントラクト(cid)へリードアクセスができることを確認します。  
(access.read_txの効果の確認）

In [7]:
var resp = await rpc.call(proxyWallet, cid, {}, { readmode: 'fast' });
console.log(resp);

{
  txno: 40297,
  txid: 'xgZiZuyXHBhAYwjQM6x5JMJSUxTYDZUjZGbmwkRZtZ9PQB',
  status: 'read',
  value: 'u86311649'
}


代理ウォレットで特定コントラクト(cid)へライトアクセスができることを確認します。
(access.write_txの効果の確認）

In [8]:
var resp = await rpc.call(proxyWallet, cid, {});
console.log(resp);

{
  txno: 40297,
  txid: 'xS5iaGysHJSEVjRYLUGgdn35C2icdevfQyVbVB6iR2zx6',
  status: 'ok',
  value: 'u86311649'
}


代理ウォレットで c1query のトランザクション検索を実行すると、取得できるのは特定コントラクト(cid)を呼び出すトランザクションのみであることを確認します。  
(access.queryの効果の確認）

In [9]:
var resp = await rpc.call(proxyWallet, 'c1query', { type: 'transactions' });
console.log(resp);

{
  txid: 'xESWQhJUchwPXc4iki3WnEViensFPMeLFUerCaKHpA9xq',
  status: 'ok',
  value: {
    list: [
      {
        txno: 40297,
        caller: [ 'u86311649', 'admin@sample_codes' ],
        callee: [ 'c026609796', 'adv15@sample_codes' ],
        status: 'ok',
        time: 1763545236825
      }
    ]
  }
}


## 代理ウォレットへの委任（リードアクセスに制限）

元ウォレットから代理ウォレットへ、限定した権限のみを委任します。  
access引数に委任する権限を指定します。read_txにunlimitedを設定します。

In [10]:
var resp = await rpc.call(adminWallet, 'c1walletproxy', {
    addr: proxyWallet.address,
    expiry: Date.now() + 3600*1000, // 委任の有効期限（現在時刻＋1時間）
    access: { read_tx: 'unlimited' },
});
console.log(resp);

{
  txno: 40298,
  txid: 'x5w3rEQGfiyzA8pRpDrDmLu7pMrZQRD8CK7Vg2fUeDCuMB',
  status: 'ok',
  value: null
}


代理ウォレットでリードアクセスができることを確認します。  

In [11]:
var resp = await rpc.call(proxyWallet, cid, {}, { readmode: 'fast' });
console.log(resp);

{
  txno: 40299,
  txid: 'xWWH74cSy42fyaQoZbBNZbwBKhBVFPRH58efv5oQi7Dq4',
  status: 'read',
  value: 'u86311649'
}


代理ウォレットでライトアクセスがエラーとなることを確認します。  

In [12]:
var resp = await rpc.call(proxyWallet, cid, {});
console.log(resp);

{
  txno: 40299,
  txid: 'xpByUFa3tD42oeiwpQ7zYoPhmgBEXvfs4v6kdGDzGMdgx',
  status: 'denied',
  value: 'walletproxy permission'
}


代理ウォレットでc1query-transactionsの問い合わせがエラーとなることを確認します。
(access.queryの効果の確認）

In [13]:
var resp = await rpc.call(proxyWallet, 'c1query', { type: 'transactions' });
console.log(resp);

{
  txid: 'xq8gc2bPw4C58gjVLYrsAuSDd7UvbaXq3jxU7Et9UE4DDB',
  status: 'denied',
  value: 'walletproxy permission'
}


## 代理ウォレットへの委任（グループを利用して動的に制限を変更）

権限の制限用にグループを作成します。

In [14]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'aclgroup', domain });
console.log(resp);
var gid = resp.value;

{
  txno: 40300,
  txid: 'xDTu7DtWyjqmqsFTvSSRwEyn8ikLYASJnRYX7Z8mBnDpKB',
  status: 'ok',
  value: 'g178802909'
}


元ウォレットから代理ウォレットへ、限定した権限のみを委任します。  
access引数に委任する権限を指定します。write_txにグループを設定します。

In [15]:
var resp = await rpc.call(adminWallet, 'c1walletproxy', {
    addr: proxyWallet.address,
    expiry: Date.now() + 3600*1000, // 委任の有効期限（現在時刻＋1時間）
    access: { write_tx: [ gid ] },
});
console.log(resp);

{
  txno: 40301,
  txid: 'xXwxe2nccND59dVz8J8gdcQ4eQihzDqkqYrj6dTULNk6t',
  status: 'ok',
  value: null
}


現在グループにはメンバーがいないため、ライトアクセスは許可されずエラーになります。

In [16]:
var resp = await rpc.call(proxyWallet, cid, {});
console.log(resp);

{
  txno: 40302,
  txid: 'xesZnE4mWDYEc78hoLLvgcQt3zqwUEKAwuiXBrF2LBV4x',
  status: 'denied',
  value: 'walletproxy permission'
}


グループにコントラクト(cid)を追加します。

In [17]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'add members', value: [ cid ] });
console.log(resp);

{
  txno: 40303,
  txid: 'xhcURhfGCeSTekBgw4kzj65mheTT2DPzJSusyb4gVnALCB',
  status: 'ok',
  value: null
}


グループにcidが追加されたので、cidへのライトアクセスは成功します。

In [18]:
var resp = await rpc.call(proxyWallet, cid, {});
console.log(resp);

{
  txno: 40304,
  txid: 'xYzhSKtxNnu29TgfixARbxzogzcjHJXYcgiDvYPVfry3u',
  status: 'ok',
  value: 'u86311649'
}


グループからコントラクト(cid)を削除します。

In [19]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'delete members', value: [ cid ] });
console.log(resp);

{
  txno: 40305,
  txid: 'xVcQxxmTCkjJ5qLtJ2Kq8iEvXPxWDW8FbPqUcHj7Lokhn',
  status: 'ok',
  value: null
}


現在グループにはメンバーがいないため、ライトアクセスは許可されずエラーになります。

In [20]:
var resp = await rpc.call(proxyWallet, cid, {});
console.log(resp);

{
  txno: 40306,
  txid: 'xTWEpVREYygzBj3S988rpGnqhXJzLgpcr8YFDPqgnkLdHB',
  status: 'denied',
  value: 'walletproxy permission'
}
